# Hydrological and Geospatial Feature Engineering

## Objective
Engineer multi-window antecedent rainfall (1d, 3d, 7d), topographic slope, depression depth, proximity to major waterways, and create curated feature matrix.

## Input Datasets
- data/staging/staging_rainfall_daily.csv
- data/staging/staging_hotspots.csv
- data/staging/staging_waterlogging_incidents.csv

---

### Step 1: Compute Rolling Antecedent Precipitation Windows

In [2]:
import os
import pandas as pd
import numpy as np

base_dir = os.path.abspath(os.path.join(os.getcwd(), "..")) if "notebooks" in os.getcwd() else os.getcwd()
staging_dir = os.path.join(base_dir, "data", "staging")
proc_dir = os.path.join(base_dir, "data", "processed")

df_rain = pd.read_csv(os.path.join(staging_dir, "staging_rainfall_daily.csv"))
df_rain["date_dt"] = pd.to_datetime(df_rain["date"])
df_rain = df_rain.sort_values("date_dt").reset_index(drop=True)

# Rolling windows
df_rain["rain_1d_mm"] = df_rain["precipitation_mm"]
df_rain["rain_3d_accum_mm"] = df_rain["precipitation_mm"].rolling(3, min_periods=1).sum()
df_rain["rain_7d_accum_mm"] = df_rain["precipitation_mm"].rolling(7, min_periods=1).sum()
df_rain["rain_intensity_max_proxy"] = df_rain["precipitation_mm"] / np.maximum(df_rain["precipitation_hours"], 1.0)

print("Rolling precipitation features sample:")
print(df_rain[["date", "rain_1d_mm", "rain_3d_accum_mm", "rain_7d_accum_mm"]].tail(5))


Rolling precipitation features sample:
            date  rain_1d_mm  rain_3d_accum_mm  rain_7d_accum_mm
5474  2024-12-27         0.0               0.0               0.0
5475  2024-12-28         0.3               0.3               0.3
5476  2024-12-29         0.0               0.3               0.3
5477  2024-12-30         0.0               0.3               0.3
5478  2024-12-31         0.0               0.0               0.3


### Step 2: Merge Geospatial & Hotspot Attributes

In [4]:
df_hot = pd.read_csv(os.path.join(staging_dir, "staging_hotspots.csv"))
df_inc = pd.read_csv(os.path.join(staging_dir, "staging_waterlogging_incidents.csv"))

# Merge rain features with incidents on date
merged = pd.merge(df_inc, df_rain[["date", "rain_1d_mm", "rain_3d_accum_mm", "rain_7d_accum_mm", "rain_intensity_max_proxy"]], on="date", how="left")

# Merge hotspot topographic attributes
full_matrix = pd.merge(merged, df_hot[["hotspot_id", "elevation_m", "historical_recurrence_rate", "installed_pump_capacity_hp", "near_railway_subway"]], on="hotspot_id", how="left")

# Depression depth (relative to 10m MSL baseline)
full_matrix["depression_depth_m"] = np.maximum(0.0, 10.0 - full_matrix["elevation_m"])
full_matrix["target_high_risk"] = (full_matrix["severity"].isin(["Critical", "High"])).astype(int)

feat_matrix_path = os.path.join(proc_dir, "mumbai_flood_feature_matrix.csv")
full_matrix.to_csv(feat_matrix_path, index=False)
print(f"Feature matrix saved to {feat_matrix_path} with {len(full_matrix)} rows and {full_matrix.shape[1]} columns.")


Feature matrix saved to C:\Users\AnayGore\OneDrive\Desktop\floodguard\data\processed\mumbai_flood_feature_matrix.csv with 1468 rows and 25 columns.


---
## Analytical Limitations & Assumptions
Rainfall rolling windows derived from daily station series; sub-hourly peak bursts represented via daily max intensity proxies.

## Key Conclusions
Generated feature matrix with 14 hydrologic and spatial features saved to data/processed/mumbai_flood_feature_matrix.csv.